# 4 · Personenregister anreichern: Konsens und Dissens

Dieses Notebook reduziert das **genderTagger-Prinzip** auf einen kleinen,
nachvollziehbaren Workflow.

Wir kombinieren vier unterschiedliche Evidenzquellen:

1. **lokale Authority-Daten**
2. **gender-guesser**
3. **nomquamgender**
4. **lokales LLM (`gpt-oss:20b`) über die TELOTA-API**

Die Methoden sehen unterschiedliche Dinge:

- Authority-Daten enthalten explizite redaktionelle Aussagen,
- Namensklassifikatoren sehen im Wesentlichen den Namen,
- das LLM sieht **Name + Kontext des Registereintrags**.


## Installation

```python
%pip install -U pandas gender-guesser nomquamgender openai
```

Der API-Key wird **nicht im Notebook gespeichert**.

Entweder vorher als Umgebungsvariable setzen:

```bash
export TELOTA_AI_API_KEY="..."
```

oder beim Ausführen interaktiv eingeben.


In [1]:
%pip install -U pandas gender-guesser nomquamgender openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import json
import math
import os
import re
from getpass import getpass

import pandas as pd
from IPython.display import display, Markdown

PERSONS_PATH = Path("data/04_register/personen_demo.csv")
AUTHORITY_PATH = Path("data/04_register/authority_demo.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = OUTPUT_DIR / "04_personenregister_angereichert.csv"

BASE_URL = os.environ.get(
    "TELOTA_AI_BASE_URL",
    "https://telota-ai-api.bbaw.de/api",
)
MODEL_ID = "gpt-oss:20b"

persons = pd.read_csv(PERSONS_PATH, sep=";")
authority = pd.read_csv(AUTHORITY_PATH, sep=";")

display(persons)
display(authority)


,id,name,note,authority_id
0,S0002266,Henriette Herz,Schriftstellerin und Gastgeberin eines Berline...,GND_HERZ
1,S0001932,Frölich,Frau des Heinrich Frölich,NaN
2,S9145415,"Klein, Auguste",Tochter des Ernst Ferdinand Klein,GND_WRONG_KLEIN
3,S3729958,"Magnus, Leopold",Genremalerin,NaN
4,S0003113,O'Bern,PredigerinHalle,NaN
5,S_DEMO_MEYER,J. Meyer,Frau des Bernhard Meyer,NaN


,authority_id,classification,note
0,GND_HERZ,female,expliziter Normdateneintrag
1,GND_WRONG_KLEIN,male,bewusst fehlerhafte Verknüpfung auf Maler Augu...


## Die Demonstrationsfälle

Die Beispiele zeigen bewusst **verschiedene Fehlertypen**:

- **Henriette Herz** – klarer Konsens
- **Frölich, Frau des Heinrich** – Name allein führt in die falsche Richtung
- **J. Meyer, Frau des Bernhard Meyer** – Initiale ist für Namensmodelle kaum verwertbar; der Kontext ist entscheidend
- **Auguste Klein** – falsche Authority-Verknüpfung
- **Leopold Magnus / O'Bern** – fehlerhafter Kontext kann gerade das LLM täuschen


## 1. Authority-Daten

Im realen Workflow wäre das beispielsweise ein GND-Lookup oder ein lokaler
Normdatenexport.

Eine Authority-Aussage ist starke Evidenz, aber auch Normdatenverknüpfungen können
fehlerhaft sein. Deshalb wird selbst diese Quelle nicht einfach blind übernommen.


In [4]:
authority_lookup = (
    authority
    .set_index("authority_id")["classification"]
    .to_dict()
)

persons["authority_result"] = persons["authority_id"].map(authority_lookup)

display(
    persons[
        ["id", "name", "authority_id", "authority_result"]
    ]
)


,id,name,authority_id,authority_result
0,S0002266,Henriette Herz,GND_HERZ,female
1,S0001932,Frölich,NaN,NaN
2,S9145415,"Klein, Auguste",GND_WRONG_KLEIN,male
3,S3729958,"Magnus, Leopold",NaN,NaN
4,S0003113,O'Bern,NaN,NaN
5,S_DEMO_MEYER,J. Meyer,NaN,NaN


## 2. gender-guesser

`gender-guesser` arbeitet listenbasiert auf Vornamen.

Das Verfahren ist billig und lokal, kennt aber **keinen Kontext**.


In [5]:
import gender_guesser.detector as gender

detector = gender.Detector(case_sensitive=False)


def first_name(full_name: str) -> str:
    value = str(full_name).strip()

    # "Klein, Auguste" -> "Auguste"
    if "," in value:
        after_comma = value.split(",", 1)[1].strip()
        return after_comma.split()[0] if after_comma else ""

    return value.split()[0] if value else ""


def map_gender_guesser(value: str):
    mapping = {
        "male": ("male", 0.90),
        "mostly_male": ("male", 0.70),
        "female": ("female", 0.90),
        "mostly_female": ("female", 0.70),
    }
    return mapping.get(value, (None, None))


persons["guesser_raw"] = persons["name"].map(
    lambda n: detector.get_gender(first_name(n))
)

mapped = persons["guesser_raw"].map(map_gender_guesser)
persons["guesser_result"] = mapped.map(lambda x: x[0])
persons["guesser_cert"] = mapped.map(lambda x: x[1])

display(
    persons[
        ["name", "guesser_raw", "guesser_result", "guesser_cert"]
    ]
)


,name,guesser_raw,guesser_result,guesser_cert
0,Henriette Herz,female,female,0.9
1,Frölich,unknown,NaN,NaN
2,"Klein, Auguste",male,male,0.9
3,"Magnus, Leopold",male,male,0.9
4,O'Bern,unknown,NaN,NaN
5,J. Meyer,unknown,NaN,NaN


## 3. nomquamgender

`nomquamgender` liefert eine Wahrscheinlichkeit `p(gf)`.

Wir erzwingen keine Entscheidung in der Mitte:

- `p(gf) ≥ 0.90` → `female`
- `p(gf) ≤ 0.10` → `male`
- dazwischen → keine automatische Zuordnung


In [6]:
import nomquamgender as nqg

nqg_model = nqg.NBGC()

given_names = persons["name"].map(first_name).tolist()
pgf_values = nqg_model.get_pgf(given_names)


def classify_pgf(value):
    if value is None:
        return None, None

    try:
        value = float(value)
    except (TypeError, ValueError):
        return None, None

    if math.isnan(value):
        return None, None

    cert = abs(value - 0.5) * 2

    if value >= 0.90:
        return "female", cert
    if value <= 0.10:
        return "male", cert

    return None, cert


classified = [classify_pgf(v) for v in pgf_values]

persons["nqg_pgf"] = pgf_values
persons["nqg_result"] = [x[0] for x in classified]
persons["nqg_cert"] = [x[1] for x in classified]

display(
    persons[
        ["name", "nqg_pgf", "nqg_result", "nqg_cert"]
    ]
)


,name,nqg_pgf,nqg_result,nqg_cert
0,Henriette Herz,0.989,female,0.978
1,Frölich,0.000,male,1.000
2,"Klein, Auguste",0.372,NaN,0.256
3,"Magnus, Leopold",0.004,male,0.992
4,O'Bern,NaN,NaN,NaN
5,J. Meyer,NaN,NaN,NaN


## 4. Lokales LLM über die TELOTA-API

Jetzt kommt eine methodisch andere Quelle hinzu.

Das LLM sieht **den vollständigen Registereintrag**, nicht nur den Namen.

Der Call geht an die OpenAI-kompatible TELOTA-Schnittstelle und verwendet
`gpt-oss:20b`.

Der Prompt zwingt das Modell auf ein kleines JSON-Schema:

```json
{
  "classification": "male | female | unknown",
  "confidence": 0.0,
  "evidence": "kurze Quellenbegründung"
}
```

Wichtig: `confidence` ist eine **Selbsteinschätzung des Modells**, keine kalibrierte
statistische Wahrscheinlichkeit.


In [7]:
from openai import OpenAI

api_key = os.environ.get("TELOTA_AI_API_KEY")

if not api_key:
    api_key = getpass("TELOTA AI API key: ")

client = OpenAI(
    base_url=BASE_URL,
    api_key=api_key,
)

print("Endpoint:", BASE_URL)
print("Modell:", MODEL_ID)


TELOTA AI API key:  ········


Endpoint: https://telota-ai-api.bbaw.de/api
Modell: gpt-oss:20b


### Ein einzelner API-Call

Diese Zelle ist absichtlich separat, damit im Vortrag sichtbar ist, wie wenig
Code für einen lokalen LLM-Aufruf nötig ist.


In [8]:
example = persons.loc[
    persons["id"] == "S0001932"
].iloc[0]

prompt = f"""
Du unterstützt die redaktionelle Anreicherung eines historischen Personenregisters.

Klassifiziere ausschließlich anhand des gegebenen Registereintrags.
Nutze kein externes Wissen.
Wenn der Eintrag keine belastbare Aussage erlaubt, verwende "unknown".

Name: {example["name"]}
Registerkontext: {example["note"]}

Antworte ausschließlich als JSON:
{{
  "classification": "male | female | unknown",
  "confidence": 0.0,
  "evidence": "kurze Begründung ausschließlich aus dem Registereintrag"
}}
""".strip()

response = client.chat.completions.create(
    model=MODEL_ID,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": (
                "Du arbeitest quellenkritisch. "
                "Erfinde keine biographischen Informationen."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ],
)

print(response.choices[0].message.content)


{"classification":"female","confidence":1.0,"evidence":"\"Frau des Heinrich Frölich\" indicates a female."}


### Alle Demo-Einträge durchlaufen

Für die eigentliche Kombination wenden wir denselben Call auf alle fünf Fälle an.


In [9]:
def parse_json_response(value: str) -> dict:
    text = (value or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    data = json.loads(text)

    classification = str(
        data.get("classification", "unknown")
    ).lower()

    if classification not in {"male", "female", "unknown"}:
        classification = "unknown"

    try:
        confidence = float(data.get("confidence"))
    except (TypeError, ValueError):
        confidence = None

    if confidence is not None:
        confidence = max(0.0, min(1.0, confidence))

    return {
        "llm_result": (
            None if classification == "unknown"
            else classification
        ),
        "llm_cert": confidence,
        "llm_evidence": str(data.get("evidence", "")).strip(),
    }


def classify_with_llm(row) -> dict:
    prompt = f"""
Du unterstützt die redaktionelle Anreicherung eines historischen Personenregisters.

Klassifiziere ausschließlich anhand des gegebenen Registereintrags.
Nutze kein externes Wissen.
Wenn der Eintrag keine belastbare Aussage erlaubt, verwende "unknown".

Name: {row["name"]}
Registerkontext: {row["note"]}

Antworte ausschließlich als JSON:
{{
  "classification": "male | female | unknown",
  "confidence": 0.0,
  "evidence": "kurze Begründung ausschließlich aus dem Registereintrag"
}}
""".strip()

    response = client.chat.completions.create(
        model=MODEL_ID,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Du arbeitest quellenkritisch. "
                    "Erfinde keine biographischen Informationen."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return parse_json_response(
        response.choices[0].message.content
    )


llm_rows = [
    classify_with_llm(row)
    for _, row in persons.iterrows()
]

llm_df = pd.DataFrame(llm_rows)
persons = pd.concat(
    [persons.reset_index(drop=True), llm_df],
    axis=1,
)

display(
    persons[
        [
            "name",
            "note",
            "llm_result",
            "llm_cert",
            "llm_evidence",
        ]
    ]
)


,name,note,llm_result,llm_cert,llm_evidence
0,Henriette Herz,Schriftstellerin und Gastgeberin eines Berline...,female,0.99,"Der Registereintrag nennt Henriette Herz als ""..."
1,Frölich,Frau des Heinrich Frölich,female,1.00,Der Registereintrag bezeichnet die Person als ...
2,"Klein, Auguste",Tochter des Ernst Ferdinand Klein,female,1.00,"Der Registerkontext nennt die Person als ""Toch..."
3,"Magnus, Leopold",Genremalerin,female,1.00,"Registerkontext: ""Genremalerin"" deutet eindeut..."
4,O'Bern,PredigerinHalle,female,0.90,"The register context ""PredigerinHalle"" contain..."
5,J. Meyer,Frau des Bernhard Meyer,female,1.00,"""Frau des Bernhard Meyer"" indicates she is the..."


## 5. Konsens und Dissens

Nun kommen vier methodisch verschiedene Quellen zusammen.

**Keine Aussage zählt nicht als Stimme.**  
Und ebenso wichtig:

> **Kein Widerspruch ist noch kein Konsens.**

Wenn nur eine einzige Methode ein Ergebnis liefert, bleibt der Fall automatisch
`low` und wird zur Prüfung vorgelegt. Erst zwei oder mehr voneinander unabhängige
Methoden können überhaupt Konsens bilden.

Das LLM ist damit **eine Stimme unter mehreren**. Seine eigene `confidence`-Angabe
ist Dokumentation des Modelloutputs, aber keine zusätzliche Evidenzquelle.


In [10]:
def aggregate(row):
    votes = {
        "authority": row["authority_result"],
        "gender-guesser": row["guesser_result"],
        "nomquamgender": row["nqg_result"],
        "gpt-oss:20b": row["llm_result"],
    }

    substantive = {
        source: value
        for source, value in votes.items()
        if isinstance(value, str)
        and value in {"male", "female"}
    }

    male_votes = sum(
        value == "male"
        for value in substantive.values()
    )
    female_votes = sum(
        value == "female"
        for value in substantive.values()
    )

    distinct = set(substantive.values())
    dissent = len(distinct) > 1
    n_sources = len(substantive)

    # Wichtig:
    # dissent=False bedeutet nur "kein Widerspruch".
    # Bei nur einer Quelle gibt es noch keinen Konsens.
    if n_sources == 0:
        suggestion = None
        cert = "low"
        status = "manuell"

    elif n_sources == 1:
        suggestion = next(iter(substantive.values()))
        cert = "low"
        status = "prüfen"

    elif male_votes == female_votes:
        suggestion = None
        cert = "low"
        status = "prüfen"

    elif dissent:
        suggestion = (
            "male"
            if male_votes > female_votes
            else "female"
        )
        cert = "low"
        status = "prüfen"

    else:
        # Mindestens zwei voneinander unabhängige Quellen stimmen überein.
        suggestion = next(iter(distinct))

        if n_sources >= 3:
            cert = "high"
        else:
            cert = "medium"

        status = "Vorschlag"

    evidence = "; ".join(
        f"{source}={value}"
        for source, value in substantive.items()
    )

    return pd.Series({
        "suggestion": suggestion,
        "cert": cert,
        "dissent": dissent,
        "sources_with_result": n_sources,
        "male_votes": male_votes,
        "female_votes": female_votes,
        "evidence": evidence,
        "review_status": status,
    })


summary = persons.apply(aggregate, axis=1)
result = pd.concat([persons, summary], axis=1)

display(
    result[
        [
            "name",
            "authority_result",
            "guesser_result",
            "nqg_result",
            "llm_result",
            "suggestion",
            "cert",
            "dissent",
            "sources_with_result",
            "review_status",
        ]
    ]
)


,name,authority_result,guesser_result,nqg_result,llm_result,suggestion,cert,dissent,sources_with_result,review_status
0,Henriette Herz,female,female,female,female,female,high,False,4,Vorschlag
1,Frölich,NaN,NaN,male,female,NaN,low,True,2,prüfen
2,"Klein, Auguste",male,male,NaN,female,male,low,True,3,prüfen
3,"Magnus, Leopold",NaN,male,male,female,male,low,True,3,prüfen
4,O'Bern,NaN,NaN,NaN,female,female,low,False,1,prüfen
5,J. Meyer,NaN,NaN,NaN,female,female,low,False,1,prüfen


## 6. Die interessanten Fälle zuerst

Für die redaktionelle Arbeit sortieren wir nach **Prüfbedarf**, nicht alphabetisch.

Genau hier wird Dissens nützlich: Er ist kein Fehler des Systems, sondern ein
**Triage-Signal**.


In [11]:
priority = {
    "prüfen": 0,
    "manuell": 1,
    "Vorschlag": 2,
}

review = result.copy()
review["_priority"] = (
    review["review_status"]
    .map(priority)
    .fillna(9)
)

review = (
    review
    .sort_values(
        [
            "dissent",
            "_priority",
            "sources_with_result",
        ],
        ascending=[False, True, True],
    )
    .drop(columns="_priority")
)

display(
    review[
        [
            "name",
            "note",
            "suggestion",
            "cert",
            "dissent",
            "evidence",
            "review_status",
        ]
    ]
)


,name,note,suggestion,cert,dissent,evidence,review_status
1,Frölich,Frau des Heinrich Frölich,NaN,low,True,nomquamgender=male; gpt-oss:20b=female,prüfen
2,"Klein, Auguste",Tochter des Ernst Ferdinand Klein,male,low,True,authority=male; gender-guesser=male; gpt-oss:2...,prüfen
3,"Magnus, Leopold",Genremalerin,male,low,True,gender-guesser=male; nomquamgender=male; gpt-o...,prüfen
4,O'Bern,PredigerinHalle,female,low,False,gpt-oss:20b=female,prüfen
5,J. Meyer,Frau des Bernhard Meyer,female,low,False,gpt-oss:20b=female,prüfen
0,Henriette Herz,Schriftstellerin und Gastgeberin eines Berline...,female,high,False,authority=female; gender-guesser=female; nomqu...,Vorschlag


## 7. CSV exportieren

Der Export enthält sowohl den aggregierten Vorschlag als auch die Einzelresultate.
Dadurch bleibt nachvollziehbar, **warum** ein Fall automatisch vorgeschlagen oder
zur Prüfung vorgelegt wurde.


In [12]:
export_columns = [
    "id",
    "name",
    "note",
    "authority_id",
    "authority_result",
    "guesser_result",
    "guesser_cert",
    "nqg_result",
    "nqg_cert",
    "llm_result",
    "llm_cert",
    "llm_evidence",
    "suggestion",
    "cert",
    "dissent",
    "male_votes",
    "female_votes",
    "sources_with_result",
    "evidence",
    "review_status",
]

review[export_columns].to_csv(
    CSV_PATH,
    sep=";",
    encoding="utf-8-sig",
    index=False,
)

print(f"CSV geschrieben: {CSV_PATH.resolve()}")


CSV geschrieben: /home/jovyan/work/ki_in_kleinen_archiven/output/04_personenregister_angereichert.csv


## Ergebnis

```text
Personenregister
      ↓
┌───────────┬────────────────┬─────────────────┬────────────────┐
│ Authority │ gender-guesser │ nomquamgender   │ gpt-oss:20b    │
└───────────┴────────────────┴─────────────────┴────────────────┘
      ↓
 Konsens / Dissens
      ↓
 Vorschlag + cert + Evidenz
      ↓
 redaktionelle Prüfung
```

Die Beispiele zeigen unterschiedliche Situationen:

- **Henriette Herz:** mehrere Quellen stimmen überein.
- **Frölich, Frau des Heinrich:** Kontext kann stärker sein als der sichtbare Männername.
- **J. Meyer:** eine Initiale reicht für Namensmodelle kaum aus; der Kontext „Frau des Bernhard Meyer“ liefert dagegen eine belastbare Spur.
- **Auguste Klein:** eine Authority-Verknüpfung kann selbst falsch sein.
- **Magnus / O'Bern:** auch ein LLM kann durch fehlerhaften Kontext getäuscht werden.

> **Uneinigkeit ist Information.**
